<a href="https://colab.research.google.com/github/JakeOh/202511_BD53/blob/main/lab_ml/ml20_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN(Recurrent Neural Network, 순환 신경망)

시계열 데이터, 자연어 처리, 자율 주행 등의 분야에 좋은 성능을 주는 신경망.

# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras

from sklearn.model_selection import train_test_split

# IMDB 데이터셋

*   imdb.com 사이트 사용자들의 영화 리뷰를 긍정(1), 부정(0)으로 분류한 데이터.
*   25,000개 훈련 샘플과 25,000개의 테스트 샘플.
*   샘플마다 단어(토큰, token)의 개수가 다름.
    *   샘플마다 특성(feature)의 개수가 다름 --> 전처리
*   keras datasets에서 다운받은 데이터는 단어들이 인코딩된 상태.
*   (참고) KoNLP 라이브러리 - 한국어 자연어 처리 라이브러리.

In [2]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=500)
# num_words=500: 가장 자주 등장하는 단어 500개를 어휘 사전으로 사용함.

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [3]:
x_train_full.shape  #> (25_000,) 모양의 1차원 배열 - 25,000개 영화 리뷰

(25000,)

In [4]:
y_train_full.shape  # (25_000,) 모양의 1차원 배열 - 25,000개 영활 리뷰의 긍정/부정

(25000,)

In [5]:
np.unique(y_train_full, return_counts=True)  # 긍정/부정 리뷰는 12,500개씩

(array([0, 1]), array([12500, 12500]))

In [7]:
x_test.shape

(25000,)

In [8]:
y_test.shape

(25000,)

In [9]:
np.unique(y_test, return_counts=True)

(array([0, 1]), array([12500, 12500]))

## 훈련 셋 탐색

In [10]:
print(x_train_full[0])  # 첫번째 훈련 샘플

[1, 14, 22, 16, 43, 2, 2, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 2, 112, 50, 2, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 2, 2, 17, 2, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2, 19, 14, 22, 4, 2, 2, 469, 4, 22, 71, 87, 12, 16, 43, 2, 38, 76, 15, 13, 2, 4, 22, 17, 2, 17, 12, 16, 2, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2, 2, 16, 480, 66, 2, 33, 4, 130, 12, 16, 38, 2, 5, 25, 124, 51, 36, 135, 48, 25, 2, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 2, 5, 2, 36, 71, 43, 2, 476, 26, 400, 317, 46, 7, 4, 2, 2, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 2, 88, 12, 16, 283, 5, 16, 2, 113, 103, 32, 15, 16, 2, 19, 178, 32]


In [11]:
print(type(x_train_full[0]))  # 첫번째 훈련 샘플 list 타입 - 정수들의 리스트

<class 'list'>


In [12]:
print(len(x_train_full[0]))  # 첫번째 훈련 샘플의 길이는 218

218


In [13]:
print(x_train_full[1])  # 두번째 훈련 샘플

[1, 194, 2, 194, 2, 78, 228, 5, 6, 2, 2, 2, 134, 26, 4, 2, 8, 118, 2, 14, 394, 20, 13, 119, 2, 189, 102, 5, 207, 110, 2, 21, 14, 69, 188, 8, 30, 23, 7, 4, 249, 126, 93, 4, 114, 9, 2, 2, 5, 2, 4, 116, 9, 35, 2, 4, 229, 9, 340, 2, 4, 118, 9, 4, 130, 2, 19, 4, 2, 5, 89, 29, 2, 46, 37, 4, 455, 9, 45, 43, 38, 2, 2, 398, 4, 2, 26, 2, 5, 163, 11, 2, 2, 4, 2, 9, 194, 2, 7, 2, 2, 349, 2, 148, 2, 2, 2, 15, 123, 125, 68, 2, 2, 15, 349, 165, 2, 98, 5, 4, 228, 9, 43, 2, 2, 15, 299, 120, 5, 120, 174, 11, 220, 175, 136, 50, 9, 2, 228, 2, 5, 2, 2, 245, 2, 5, 4, 2, 131, 152, 491, 18, 2, 32, 2, 2, 14, 9, 6, 371, 78, 22, 2, 64, 2, 9, 8, 168, 145, 23, 4, 2, 15, 16, 4, 2, 5, 28, 6, 52, 154, 462, 33, 89, 78, 285, 16, 145, 95]


In [14]:
print(len(x_train_full[1]))  # 두번째 훈련 샘플의 길이는 189

189


imdb 데이터셋은 파이썬의 list 객체들을 원소로 갖는 **1차원 ndarray**.

각각의 리스트들은 정수들을 저장. 정수들은 영화 리뷰에 사용된 단어(토큰)를 의미.

각각의 리스트들은 길이가 다름.

In [15]:
for i in range(5):
    print(f'인덱스 {i}: 토큰 개수 = {len(x_train_full[i])}')

인덱스 0: 토큰 개수 = 218
인덱스 1: 토큰 개수 = 189
인덱스 2: 토큰 개수 = 141
인덱스 3: 토큰 개수 = 550
인덱스 4: 토큰 개수 = 147


In [18]:
for i in range(5):
    print(x_train_full[i][:30])

[1, 14, 22, 16, 43, 2, 2, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 2, 112, 50, 2, 2, 9, 35, 480]
[1, 194, 2, 194, 2, 78, 228, 5, 6, 2, 2, 2, 134, 26, 4, 2, 8, 118, 2, 14, 394, 20, 13, 119, 2, 189, 102, 5, 207, 110]
[1, 14, 47, 8, 30, 31, 7, 4, 249, 108, 7, 4, 2, 54, 61, 369, 13, 71, 149, 14, 22, 112, 4, 2, 311, 12, 16, 2, 33, 75]
[1, 4, 2, 2, 33, 2, 4, 2, 432, 111, 153, 103, 4, 2, 13, 70, 131, 67, 11, 61, 2, 2, 35, 2, 2, 61, 2, 452, 2, 4]
[1, 249, 2, 7, 61, 113, 10, 10, 13, 2, 14, 20, 56, 33, 2, 18, 457, 88, 13, 2, 2, 45, 2, 13, 70, 79, 49, 2, 2, 13]


## 단어 인덱스(word index)

In [19]:
# get_word_index() 함수: 단어를 키로 하고, 그 단어의 인덱스(빈도수 순위)를 값으로 갖는 dict를 다운로드.
# 영화 리뷰의 단어들을 숫자로 인코딩할 때 인덱스들을 이용하기 위해서 만들어진 사전.
# 인덱스가 작을 수록 빈도수 순위가 높음.
word_index = keras.datasets.imdb.get_word_index()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [20]:
print(type(word_index))

<class 'dict'>


In [24]:
for i, (k, v) in enumerate(word_index.items()):
    print(k, ':', v)
    if i == 9:
        break

fawn : 34701
tsukino : 52006
nunnery : 52007
sonja : 16816
vani : 63951
woods : 1408
spiders : 16115
hanging : 2345
woody : 2289
trawling : 52008


word_index를 인덱스를 키로 하고, 단어를 값으로 갖는 dict로 변환.

*   0: padding(패딩)
*   1: 문장의 시작
*   2: load_data() 함수의 아규먼트 num_words에 포함되지 않은 단어들. n개의 어휘 사전에 포함되지 않은 단어.

In [26]:
start_idx = 1  # 문장의 시작
oov_idx = 2  # n개의 어휘 사전에 없는 단어(out of value)
idx_from = 3
index_word = {(v + idx_from):k for k, v in word_index.items()}

In [27]:
index_word[start_idx] = '[START]'
index_word[oov_idx] = '[OOV]'

In [32]:
for i in range(1, 11):
    print(i, ':', index_word.get(i))

1 : [START]
2 : [OOV]
3 : None
4 : the
5 : and
6 : a
7 : of
8 : to
9 : is
10 : br


In [29]:
# 샘플의 인코딩된 숫자들을 단어로 변환하는 함수
def decode_review(review):
    # review: 숫자들의 리스트
    return ' '.join([index_word.get(i) for i in review])

In [30]:
decode_review(x_train_full[0])

"[START] this film was just [OOV] [OOV] [OOV] [OOV] story direction [OOV] really [OOV] the part they played and you could just [OOV] being there [OOV] [OOV] is an amazing actor and now the same being director [OOV] father came from the same [OOV] [OOV] as [OOV] so i loved the fact there was a real [OOV] with this film the [OOV] [OOV] throughout the film were great it was just [OOV] so much that i [OOV] the film as [OOV] as it was [OOV] for [OOV] and would recommend it to everyone to watch and the [OOV] [OOV] was amazing really [OOV] at the end it was so [OOV] and you know what they say if you [OOV] at a film it must have been good and this definitely was also [OOV] to the two little [OOV] that played the [OOV] of [OOV] and [OOV] they were just [OOV] children are often left out of the [OOV] [OOV] i think because the stars that play them all [OOV] up are such a big [OOV] for the whole film but these children are amazing and should be [OOV] for what they have done don't you think the whol

In [31]:
decode_review(x_train_full[1])

"[START] big [OOV] big [OOV] bad music and a [OOV] [OOV] [OOV] these are the [OOV] to best [OOV] this terrible movie i love [OOV] horror movies and i've seen [OOV] but this had got to be on of the worst ever made the plot is [OOV] [OOV] and [OOV] the acting is an [OOV] the script is completely [OOV] the best is the end [OOV] with the [OOV] and how he [OOV] out who the killer is it's just so [OOV] [OOV] written the [OOV] are [OOV] and funny in [OOV] [OOV] the [OOV] is big [OOV] of [OOV] [OOV] men [OOV] those [OOV] [OOV] [OOV] that show off their [OOV] [OOV] that men actually [OOV] them and the music is just [OOV] [OOV] that plays over and over again in almost every scene there is [OOV] music [OOV] and [OOV] [OOV] away [OOV] and the [OOV] still doesn't close for [OOV] all [OOV] [OOV] this is a truly bad film [OOV] only [OOV] is to look back on the [OOV] that was the [OOV] and have a good old laugh at how bad everything was back then"